# RL vs Uniform Sampling for Collocation Points

> **Repository:** [PINNs-RL-PDE](https://github.com/josegarciav/PINNs-RL-PDE) &nbsp;|&nbsp; **Package:** `pinnrl` &nbsp;|&nbsp; **Estimated run time:** ~5 minutes on CPU

PINNs minimise a physics loss evaluated at **collocation points** — randomly sampled coordinates in the
spatio-temporal domain.  Where those points land matters enormously: waste points on smooth regions
and the network has no signal about sharp features like shocks or boundary layers.

This notebook compares two strategies head-to-head:

| Strategy | How it works |
|----------|-------------|
| **Uniform** | Draw $(x, t)$ from $\mathcal{U}([x_{\min}, x_{\max}] \times [t_{\min}, t_{\max}])$ every epoch — the baseline |
| **RL-guided** | A DQN agent observes the current residual field and outputs a sampling probability map that concentrates points where the PDE residual is highest |

We test on the **Burgers equation** ($\nu = 0.01$), which develops a near-discontinuous shock and
therefore benefits most from adaptive point placement.

---

## The Hypothesis

Uniform sampling wastes ~50 % of collocation points on smooth regions.  An RL agent that
learns to concentrate points near the shock front should:

1. Achieve **lower PDE residual** for the same point budget
2. Produce a **more accurate solution** near the shock
3. Show a clear **density shift** in point placement over training

## 1  Setup & Imports

In [ ]:
import sys, os, time, copy

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import torch
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")

device = torch.device("cpu")
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"PyTorch  : {torch.__version__}")
print(f"Device   : {device}")

## 2  Configure the Burgers Equation

$$\frac{\partial u}{\partial t} + u\frac{\partial u}{\partial x} = \nu\frac{\partial^2 u}{\partial x^2}, \quad x\in[-1,1],\; t\in[0,1],\; \nu = 0.01$$

This is the same setup as notebook 02.  The near-inviscid regime ($\nu = 0.01$) produces a
shock around $t \approx 1/\pi$, making it the ideal test case for adaptive sampling.

In [ ]:
from src.pdes.pde_base import PDEConfig
from src.pdes.burgers_equation import BurgersEquation

NU = 0.01
X_MIN, X_MAX = -1.0, 1.0
T_MIN, T_MAX =  0.0, 1.0

def make_burgers_pde():
    """Create a fresh BurgersEquation instance."""
    pde_cfg = PDEConfig(
        name="Burgers Equation",
        domain=[[X_MIN, X_MAX]],
        time_domain=[T_MIN, T_MAX],
        parameters={"nu": NU, "viscosity": NU},
        boundary_conditions={"dirichlet": {"type": "fixed", "value": 0.0}},
        initial_condition={"type": "sine", "amplitude": -1.0, "frequency": 1.0},
        exact_solution={
            "type": "cole_hopf",
            "viscosity": NU,
            "initial_amplitude": -1.0,
            "initial_frequency": 1.0,
        },
        dimension=1,
        input_dim=2,
        output_dim=1,
        device=device,
        training={
            "num_collocation_points": 1000,
            "num_boundary_points": 100,
            "num_initial_points": 100,
            "loss_weights": {"residual": 1.0, "boundary": 10.0,
                             "initial": 10.0, "smoothness": 0.0},
        },
    )
    return BurgersEquation(pde_cfg)

print(f"Burgers Equation  nu={NU}")
print(f"Domain: x in [{X_MIN}, {X_MAX}], t in [{T_MIN}, {T_MAX}]")

## 3  Build Identical Models for Both Strategies

Both runs use the **same architecture** (Fourier features, 64-wide, 3 layers) initialised
from the **same weights** so the only variable is the sampling strategy.

In [ ]:
from src.config import ModelConfig
from src.neural_networks import PINNModel

class _Cfg:
    """Lightweight config wrapper for PINNModel."""
    def __init__(self, model_cfg, dev):
        self.model  = model_cfg
        self.device = dev

def build_model():
    """Build a Fourier-features PINN."""
    cfg = ModelConfig(
        input_dim=2,
        hidden_dim=64,
        output_dim=1,
        num_layers=3,
        activation="tanh",
        architecture="fourier",
    )
    cfg.mapping_size = 32
    cfg.scale = 4.0
    cfg.device = device
    return PINNModel(_Cfg(cfg, device), device=device).to(device)

# Build a base model and clone it so both start from identical weights
torch.manual_seed(SEED)
base_model = build_model()
init_state = copy.deepcopy(base_model.state_dict())

n_params = sum(p.numel() for p in base_model.parameters() if p.requires_grad)
print(f"Architecture : Fourier Features")
print(f"Parameters   : {n_params:,}")

## 4  The RL Agent

`pinnrl` ships a **DQN-based RL agent** (`RLAgent`) that:

1. Observes a **state** — a summary of the current residual field
2. Outputs an **action** — a probability map over a discretised $(x, t)$ grid
3. Receives a **reward** $r = -(w_r L_r + w_b L_b + w_i L_i)$ — negative weighted loss

We configure a lightweight agent (64-dim hidden, 50-cell grid) so the notebook runs quickly.
The agent uses **epsilon-greedy** exploration that anneals from 1.0 → 0.1 over training.

In [ ]:
from src.rl.rl_agent import RLAgent

def make_rl_agent():
    """Create a DQN agent for adaptive collocation sampling."""
    return RLAgent(
        state_dim=4,           # [mean_residual, max_residual, mean_boundary, epoch_frac]
        action_dim=50,         # 50-cell 1-D grid discretisation
        hidden_dim=64,
        learning_rate=1e-3,
        gamma=0.99,
        epsilon_start=1.0,
        epsilon_end=0.1,
        epsilon_decay=0.995,
        memory_size=2000,
        batch_size=32,
        target_update=50,
        reward_weights={
            "residual": 1.0,
            "boundary": 1.0,
            "initial": 1.0,
            "exploration": 0.1,
        },
        device=device,
    )

print("RL agent: DQN, 64-hidden, 50-action, eps 1.0 -> 0.1")

## 5  RL-Guided Sampling Function

The key idea: after each epoch the agent sees the residual statistics, picks an action
(which region to oversample), and we bias the collocation distribution accordingly.

We split the domain into `action_dim` cells.  The agent's Q-values determine how many
points each cell receives — higher Q → more points.

In [ ]:
def rl_guided_sample(agent, state, n_points, epoch):
    """
    Sample collocation points biased by the RL agent's Q-values.

    The agent produces Q-values over a 1-D grid of cells.  We convert
    these to a categorical distribution and draw spatial locations from
    the selected cells, with uniform time sampling.
    """
    with torch.no_grad():
        q_values = agent.policy_net(state.unsqueeze(0)).squeeze(0)  # (action_dim,)

    # Convert Q-values to a sampling distribution via softmax
    probs = torch.softmax(q_values, dim=0)

    # Draw cell indices according to the distribution
    cell_indices = torch.multinomial(probs, n_points, replacement=True)

    # Map cell indices to x-coordinates (centre of each cell + small jitter)
    n_cells = agent.action_dim
    cell_width = (X_MAX - X_MIN) / n_cells
    x_centres = X_MIN + (cell_indices.float() + 0.5) * cell_width
    jitter = (torch.rand(n_points, device=device) - 0.5) * cell_width
    x = (x_centres + jitter).clamp(X_MIN, X_MAX).unsqueeze(1)

    # Time is sampled uniformly (the agent focuses on spatial placement)
    t = torch.rand(n_points, 1, device=device) * (T_MAX - T_MIN) + T_MIN

    return x, t, probs.detach().cpu().numpy()

## 6  Training Loop — Both Strategies

We train two copies of the same model for **200 epochs** with **500 collocation points** per epoch.
The only difference is where those 500 points are placed.

In [ ]:
N_EPOCHS = 200
N_COLL   = 500
LR       = 5e-3

def compute_state(pde, model, epoch, n_epochs):
    """Compute a compact state vector from the current residual field."""
    model.eval()
    with torch.no_grad():
        x_probe = torch.linspace(X_MIN, X_MAX, 100, device=device).unsqueeze(1)
        t_probe = torch.rand(100, 1, device=device) * (T_MAX - T_MIN) + T_MIN
    # Need gradients for residual computation
    x_probe = x_probe.detach().requires_grad_(True)
    t_probe = t_probe.detach().requires_grad_(True)
    model.train()
    residual = pde.compute_residual(model, x_probe, t_probe)
    res_abs = residual.abs().detach()
    return torch.tensor([
        res_abs.mean().item(),
        res_abs.max().item(),
        res_abs.std().item(),
        epoch / n_epochs,
    ], device=device)


def train_one_strategy(strategy_name, use_rl=False):
    """Train a model using the specified sampling strategy."""
    print(f"\n{'='*55}")
    print(f"  Training: {strategy_name}")
    print(f"{'='*55}")

    pde   = make_burgers_pde()
    model = build_model()
    model.load_state_dict(copy.deepcopy(init_state))  # Same starting weights
    opt   = torch.optim.Adam(model.parameters(), lr=LR)

    agent = make_rl_agent() if use_rl else None
    prev_state = None

    loss_history = []
    res_history  = []  # track residual loss separately
    point_snapshots = []  # store (x, t) snapshots for visualisation
    density_snapshots = []  # RL sampling density snapshots
    t0 = time.time()

    for epoch in range(1, N_EPOCHS + 1):
        model.train()
        opt.zero_grad()

        # ── Sample collocation points ───────────────────────────────────
        if use_rl and agent is not None:
            state = compute_state(pde, model, epoch, N_EPOCHS)
            x_c, t_c, density = rl_guided_sample(agent, state, N_COLL, epoch)
            if epoch in (1, 50, 100, 150, 200):
                density_snapshots.append((epoch, density.copy()))
        else:
            x_c = torch.rand(N_COLL, 1, device=device) * (X_MAX - X_MIN) + X_MIN
            t_c = torch.rand(N_COLL, 1, device=device) * (T_MAX - T_MIN) + T_MIN

        # Save point snapshot at key epochs
        if epoch in (1, 50, 100, 150, 200):
            point_snapshots.append(
                (epoch, x_c.detach().cpu().numpy(), t_c.detach().cpu().numpy())
            )

        # ── Compute loss ────────────────────────────────────────────────
        try:
            losses = pde.compute_loss(model, x_c, t_c)
            total  = losses["total"]
            if torch.isnan(total) or torch.isinf(total):
                loss_history.append(float("nan"))
                res_history.append(float("nan"))
                continue
            total.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            loss_history.append(total.item())
            res_history.append(losses["residual"].item())
        except Exception as exc:
            print(f"  Epoch {epoch:>3}: error — {exc}")
            loss_history.append(float("nan"))
            res_history.append(float("nan"))
            continue

        # ── RL agent update ─────────────────────────────────────────────
        if use_rl and agent is not None:
            reward = agent.compute_reward(
                losses["residual"].item(),
                losses["boundary"].item(),
                losses["initial"].item(),
            )
            new_state = compute_state(pde, model, epoch, N_EPOCHS)
            done = (epoch == N_EPOCHS)
            if prev_state is not None:
                agent.update(prev_state, 0, reward, new_state, done)
            prev_state = new_state

        if epoch % 50 == 0 or epoch == 1:
            print(f"  Epoch {epoch:>3}: loss = {total.item():.4e}")

    elapsed = time.time() - t0
    print(f"  Done in {elapsed:.1f}s  |  Final loss: {loss_history[-1]:.4e}")

    return {
        "model": model,
        "loss_history": loss_history,
        "res_history": res_history,
        "time": elapsed,
        "point_snapshots": point_snapshots,
        "density_snapshots": density_snapshots,
        "agent": agent,
    }


# ── Run both ────────────────────────────────────────────────────────────
uniform_result = train_one_strategy("Uniform Sampling",   use_rl=False)
rl_result      = train_one_strategy("RL-Guided Sampling", use_rl=True)

## 7  Loss Curves — Uniform vs RL

In [ ]:
COLORS = {"uniform": "#1f77b4", "rl": "#d62728"}

def smooth(arr, window=7):
    """Rolling mean for smoother curves."""
    if len(arr) < window:
        return arr
    return np.convolve(arr, np.ones(window) / window, mode="valid")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Total loss
ax = axes[0]
u_smooth = smooth(uniform_result["loss_history"])
r_smooth = smooth(rl_result["loss_history"])
ax.semilogy(range(1, len(u_smooth) + 1), u_smooth, lw=2, color=COLORS["uniform"], label="Uniform")
ax.semilogy(range(1, len(r_smooth) + 1), r_smooth, lw=2, color=COLORS["rl"], label="RL-guided")
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Total Loss (log)", fontsize=12)
ax.set_title("Total Training Loss", fontsize=13, fontweight="bold")
ax.legend(fontsize=11)

# Residual loss only
ax = axes[1]
u_res = smooth(uniform_result["res_history"])
r_res = smooth(rl_result["res_history"])
ax.semilogy(range(1, len(u_res) + 1), u_res, lw=2, color=COLORS["uniform"], label="Uniform")
ax.semilogy(range(1, len(r_res) + 1), r_res, lw=2, color=COLORS["rl"], label="RL-guided")
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("PDE Residual Loss (log)", fontsize=12)
ax.set_title("PDE Residual Loss", fontsize=13, fontweight="bold")
ax.legend(fontsize=11)

fig.suptitle("Uniform vs RL-Guided Sampling — Burgers Equation",
             fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()

print(f"\nFinal total loss — Uniform: {uniform_result['loss_history'][-1]:.4e}")
print(f"Final total loss — RL:      {rl_result['loss_history'][-1]:.4e}")

## 8  Collocation Point Distribution Over Training

The key visual: how do the collocation points shift as training progresses?

- **Uniform** — points remain scattered everywhere
- **RL** — points should concentrate near $x \approx 0$ where the shock forms

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7), sharey=True, sharex=True)

for row, (result, label) in enumerate(
    [(uniform_result, "Uniform"), (rl_result, "RL-guided")]
):
    for col, (epoch, x_pts, t_pts) in enumerate(result["point_snapshots"]):
        ax = axes[row, col]
        ax.scatter(x_pts, t_pts, s=2, alpha=0.4,
                   color=COLORS["uniform" if row == 0 else "rl"])
        ax.set_title(f"Epoch {epoch}", fontsize=10)
        if col == 0:
            ax.set_ylabel(label, fontsize=12, fontweight="bold")
        if row == 1:
            ax.set_xlabel("x", fontsize=10)
        ax.set_xlim(X_MIN, X_MAX)
        ax.set_ylim(T_MIN, T_MAX)
        # Mark approximate shock location
        ax.axvline(x=0, color="gray", ls="--", lw=0.8, alpha=0.5)

fig.suptitle("Collocation Point Distribution Over Training",
             fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()

## 9  RL Sampling Density Evolution

The DQN agent's softmax output tells us its learned sampling preference across the spatial
domain.  Early on it is near-uniform (high epsilon → random exploration).  As training
progresses it should develop a peak near the shock region.

In [ ]:
if rl_result["density_snapshots"]:
    fig, ax = plt.subplots(figsize=(10, 5))

    n_cells = len(rl_result["density_snapshots"][0][1])
    x_cells = np.linspace(X_MIN, X_MAX, n_cells)

    cmap = plt.cm.viridis
    n_snaps = len(rl_result["density_snapshots"])

    for i, (epoch, density) in enumerate(rl_result["density_snapshots"]):
        color = cmap(i / max(1, n_snaps - 1))
        ax.plot(x_cells, density, lw=1.8, color=color, label=f"Epoch {epoch}")

    ax.axhline(y=1.0 / n_cells, color="gray", ls="--", lw=1, label="Uniform baseline")
    ax.axvline(x=0, color="red", ls=":", lw=1, alpha=0.6, label="Shock location")

    ax.set_xlabel("x", fontsize=12)
    ax.set_ylabel("Sampling probability", fontsize=12)
    ax.set_title("RL Agent Sampling Density Evolution",
                 fontsize=13, fontweight="bold")
    ax.legend(fontsize=9)
    fig.tight_layout()
    plt.show()
else:
    print("No density snapshots recorded.")

## 10  Solution Comparison at t = 0.5

The shock in Burgers' equation is fully formed by $t = 0.5$.  We compare
both models against the exact Cole-Hopf solution at this snapshot.

In [ ]:
T_SNAP = 0.5
N_PLOT = 300

x_plot = torch.linspace(X_MIN, X_MAX, N_PLOT, device=device).unsqueeze(1)
t_plot = torch.full((N_PLOT, 1), T_SNAP, device=device)
x_np   = x_plot.cpu().numpy().ravel()

# Cole-Hopf exact solution
def cole_hopf_exact(x, t, nu=NU):
    """Exact Burgers solution via Cole-Hopf."""
    phi  = -np.cos(np.pi * x) * np.exp(-nu * np.pi**2 * t)
    dphi =  np.pi * np.sin(np.pi * x) * np.exp(-nu * np.pi**2 * t)
    eps  = 1e-8
    phi_safe = np.where(np.abs(phi) < eps, np.sign(phi) * eps, phi)
    return -2.0 * nu * dphi / phi_safe

u_exact = cole_hopf_exact(x_np, T_SNAP)

def predict(model):
    """Run inference on the (x, t) grid."""
    model.eval()
    with torch.no_grad():
        return model(torch.cat([x_plot, t_plot], dim=1)).cpu().numpy().ravel()

u_uniform = predict(uniform_result["model"])
u_rl      = predict(rl_result["model"])

# ── Plot ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Left: solution profiles
ax = axes[0]
ax.plot(x_np, u_exact,   lw=2.5, color="black", label="Exact")
ax.plot(x_np, u_uniform, lw=1.8, ls="--", color=COLORS["uniform"], label="Uniform")
ax.plot(x_np, u_rl,      lw=1.8, ls="--", color=COLORS["rl"],      label="RL-guided")
ax.set_xlabel("x", fontsize=12)
ax.set_ylabel(f"u(x, t={T_SNAP})", fontsize=12)
ax.set_title("Solution Profiles", fontsize=12, fontweight="bold")
ax.legend(fontsize=10)
ax.set_xlim(X_MIN, X_MAX)

# Middle: pointwise error
ax = axes[1]
err_uniform = np.abs(u_uniform - u_exact)
err_rl      = np.abs(u_rl - u_exact)
ax.plot(x_np, err_uniform, lw=1.5, color=COLORS["uniform"], label="Uniform")
ax.plot(x_np, err_rl,      lw=1.5, color=COLORS["rl"],      label="RL-guided")
ax.set_xlabel("x", fontsize=12)
ax.set_ylabel("|u_pred - u_exact|", fontsize=12)
ax.set_title("Pointwise Absolute Error", fontsize=12, fontweight="bold")
ax.legend(fontsize=10)
ax.set_xlim(X_MIN, X_MAX)

# Right: bar chart of L2 errors
ax = axes[2]
l2_uniform = np.sqrt(np.mean((u_uniform - u_exact)**2))
l2_rl      = np.sqrt(np.mean((u_rl - u_exact)**2))
bars = ax.bar(["Uniform", "RL-guided"], [l2_uniform, l2_rl],
              color=[COLORS["uniform"], COLORS["rl"]],
              edgecolor="white", width=0.5)
for bar, val in zip(bars, [l2_uniform, l2_rl]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.05,
            f"{val:.3e}", ha="center", fontsize=10)
ax.set_ylabel("L2 Error", fontsize=12)
ax.set_title("L2 Error Comparison", fontsize=12, fontweight="bold")

fig.suptitle(f"Uniform vs RL — Burgers Equation at t = {T_SNAP}",
             fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()

improvement = (l2_uniform - l2_rl) / l2_uniform * 100
print(f"\nL2 error — Uniform:   {l2_uniform:.4e}")
print(f"L2 error — RL-guided: {l2_rl:.4e}")
print(f"Improvement:          {improvement:+.1f}%")

## 11  Space-Time Error Heat Maps

A global view of where each strategy succeeds and fails across the full $(x, t)$ domain.

In [ ]:
NX, NT = 100, 100
x_grid = np.linspace(X_MIN, X_MAX, NX)
t_grid = np.linspace(T_MIN, T_MAX, NT)
XX, TT = np.meshgrid(x_grid, t_grid)

xt_flat = torch.tensor(
    np.stack([XX.ravel(), TT.ravel()], axis=1), dtype=torch.float32, device=device
)

# Exact solution on the grid
u_exact_grid = cole_hopf_exact(XX, TT)

# Predictions
uniform_result["model"].eval()
rl_result["model"].eval()

with torch.no_grad():
    u_uniform_grid = uniform_result["model"](xt_flat).cpu().numpy().reshape(NT, NX)
    u_rl_grid      = rl_result["model"](xt_flat).cpu().numpy().reshape(NT, NX)

err_uniform_grid = np.abs(u_uniform_grid - u_exact_grid)
err_rl_grid      = np.abs(u_rl_grid - u_exact_grid)

vmax = max(err_uniform_grid.max(), err_rl_grid.max())

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
kw = dict(extent=[X_MIN, X_MAX, T_MIN, T_MAX], origin="lower",
          aspect="auto", cmap="hot_r", vmin=0, vmax=vmax)

im0 = axes[0].imshow(err_uniform_grid, **kw)
axes[0].set_title("Uniform — |error|", fontweight="bold")
axes[0].set_xlabel("x"); axes[0].set_ylabel("t")
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(err_rl_grid, **kw)
axes[1].set_title("RL-guided — |error|", fontweight="bold")
axes[1].set_xlabel("x"); axes[1].set_ylabel("t")
plt.colorbar(im1, ax=axes[1])

# Difference: positive = RL is better
diff = err_uniform_grid - err_rl_grid
vlim = max(abs(diff.min()), abs(diff.max()))
im2 = axes[2].imshow(diff, extent=[X_MIN, X_MAX, T_MIN, T_MAX],
                     origin="lower", aspect="auto", cmap="RdBu_r",
                     vmin=-vlim, vmax=vlim)
axes[2].set_title("Difference (blue = RL better)", fontweight="bold")
axes[2].set_xlabel("x"); axes[2].set_ylabel("t")
plt.colorbar(im2, ax=axes[2])

fig.suptitle("Space-Time Error Maps — Uniform vs RL",
             fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()

## 12  Summary Table

In [ ]:
print(f"\n{'Strategy':<18} {'Final Loss':>12} {'L2 Error':>12} {'Time (s)':>10}")
print("-" * 56)
print(f"  {'Uniform':<16} {uniform_result['loss_history'][-1]:>12.4e} {l2_uniform:>12.4e} {uniform_result['time']:>10.1f}")
print(f"  {'RL-guided':<16} {rl_result['loss_history'][-1]:>12.4e} {l2_rl:>12.4e} {rl_result['time']:>10.1f}")
print(f"\nRL overhead: {rl_result['time'] - uniform_result['time']:.1f}s "
      f"({(rl_result['time'] / uniform_result['time'] - 1) * 100:.0f}% slower)")

## 13  Save Key Plots

In [ ]:
IMAGES_DIR = os.path.join(os.getcwd(), "images")
os.makedirs(IMAGES_DIR, exist_ok=True)

# Save the solution comparison
fig_save, axes_save = plt.subplots(1, 3, figsize=(16, 5))

ax = axes_save[0]
ax.plot(x_np, u_exact,   lw=2.5, color="black", label="Exact")
ax.plot(x_np, u_uniform, lw=1.8, ls="--", color=COLORS["uniform"], label="Uniform")
ax.plot(x_np, u_rl,      lw=1.8, ls="--", color=COLORS["rl"],      label="RL-guided")
ax.set_xlabel("x"); ax.set_ylabel(f"u(x, t={T_SNAP})")
ax.set_title("Solution Profiles", fontweight="bold")
ax.legend(fontsize=9); ax.set_xlim(X_MIN, X_MAX)

ax = axes_save[1]
ax.plot(x_np, err_uniform, lw=1.5, color=COLORS["uniform"], label="Uniform")
ax.plot(x_np, err_rl,      lw=1.5, color=COLORS["rl"],      label="RL-guided")
ax.set_xlabel("x"); ax.set_ylabel("|error|")
ax.set_title("Pointwise Error", fontweight="bold")
ax.legend(fontsize=9); ax.set_xlim(X_MIN, X_MAX)

ax = axes_save[2]
ax.bar(["Uniform", "RL"], [l2_uniform, l2_rl],
       color=[COLORS["uniform"], COLORS["rl"]], width=0.5)
ax.set_ylabel("L2 Error"); ax.set_title("L2 Comparison", fontweight="bold")

fig_save.suptitle(f"RL vs Uniform Sampling — Burgers (t={T_SNAP})", fontweight="bold")
fig_save.tight_layout()
save_path = os.path.join(IMAGES_DIR, "03_rl_vs_uniform_solution.png")
fig_save.savefig(save_path, dpi=150, bbox_inches="tight")
plt.close(fig_save)
print(f"Saved: {save_path}")

## 14  Analysis: When Does RL Help?

### Where RL shines

- **Sharp features** — shocks (Burgers, KdV), boundary layers, phase boundaries
  (Allen-Cahn, Cahn-Hilliard).  These are exactly the regions where wasting collocation
  points on smooth areas hurts most.
- **Long training runs** — the RL agent needs ~50–100 epochs of exploration before its
  policy becomes useful.  On very short runs the overhead dominates.
- **Fixed point budgets** — when compute is limited and you can't just add more points,
  placing them intelligently matters.

### Where uniform is fine

- **Smooth solutions** — heat equation with moderate diffusivity, simple convection.
  The PDE residual is roughly uniform, so there's no benefit to biasing.
- **Very short runs** (<50 epochs) — the agent hasn't learned anything yet and
  adds computational overhead.
- **Very large point budgets** — with enough points, uniform sampling covers
  the domain densely enough that adaptive placement offers diminishing returns.

### The cost of RL

The DQN adds per-epoch overhead for:
1. Computing the state vector (one extra forward pass on a probe grid)
2. Forward pass through the DQN policy network
3. Replay buffer storage and periodic training updates

In this notebook the RL run is typically 30–60 % slower per epoch.  The question is
whether the accuracy improvement justifies this cost.

### Recommendation

| Scenario | Use RL? |
|----------|--------|
| Burgers / KdV / Allen-Cahn with shocks | **Yes** |
| Heat / convection (smooth) | No — uniform is sufficient |
| Quick prototyping (<100 epochs) | No — overhead not recovered |
| Production run (>1000 epochs, fixed budget) | **Yes** — accuracy gains compound |

---

## 15  Next Steps

- **Increase `N_EPOCHS`** to 500+ and watch the RL advantage grow as the agent's policy matures.
- **Try other PDEs** — Allen-Cahn and Cahn-Hilliard have phase interfaces that RL can target.
- **Compare with RAR** (Residual-Adaptive Refinement) — a non-learned strategy that adds points
  where the residual is large.  This is planned for v0.3.
- **Tune the RL agent** — `epsilon_decay`, `hidden_dim`, `action_dim` all affect performance.
  The dashboard's collocation tab lets you visualise the learned density in real time.

---

**Previous notebook:** [`02_comparing_architectures.ipynb`](02_comparing_architectures.ipynb)